# Generate Sample Data for ADLS Gen2

**Purpose:** Generate sample AXA placement data in ADLS Gen2 for Day 2 E2E Pipeline Labs.

---

## Overview

This notebook creates sample data for the Day 2 End-to-End Pipeline labs which use ADLS Gen2 storage.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           DATA GENERATION OVERVIEW                          │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   This notebook generates:                                                  │
│                                                                             │
│   1. AXA Entities Reference Data (18 records)                              │
│      -> Used for enrichment joins                                          │
│                                                                             │
│   2. Investment Placements - Initial Load (300 records)                    │
│      -> Used for batch ingestion in Bronze Layer Lab                       │
│                                                                             │
│   3. Investment Placements - Incremental Batches (50 + 30 records)         │
│      -> Used for Auto Loader demo in Bronze Layer Lab                      │
│                                                                             │
│   Output Format: JSON files in ADLS Gen2                                   │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

---

## Data Usage in Labs

| Dataset | Records | Format | Used In |
|---------|---------|--------|----------|
| AXA Entities | 18 | JSON | Silver Layer - Enrichment |
| Placements Initial | 300 | JSON | Bronze Layer - Batch Ingestion |
| Placements Incremental | 80 | JSON | Bronze Layer - Auto Loader |

---

## Configuration

Update the values below to match your Azure environment.

In [ ]:
# ============================================================
# CONFIGURATION (UC External Location handles ADLS auth)
# ============================================================

STORAGE_ACCOUNT = dbutils.secrets.get(scope="training", key="storage-account-name")
CONTAINER = dbutils.secrets.get(scope="training", key="container")
WS_KEY = dbutils.secrets.get(scope="training", key="ws-key")

BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{WS_KEY}"
RAW_PATH = f"{BASE_PATH}/raw"
ENTITIES_PATH = f"{RAW_PATH}/entities"
PLACEMENTS_PATH = f"{RAW_PATH}/placements"
PLACEMENTS_INCREMENTAL_PATH = f"{RAW_PATH}/placements_incremental"

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Storage Account: {STORAGE_ACCOUNT}")
print(f"Workspace Key:   {WS_KEY}")
print(f"Base Path:       {BASE_PATH}")

In [ ]:
# Imports
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType
from pyspark.sql.functions import (
    col, lit, expr, rand, round as spark_round, 
    to_date, date_add, current_timestamp, concat, lpad,
    array, element_at, floor
)
import uuid

print("Libraries imported successfully!")

---

## 1. Generate AXA Entities Reference Data

Reference table for AXA Group companies worldwide. Used for enrichment joins in Silver Layer.

In [ ]:
# AXA Entities - Reference data for enrichment

entities_schema = StructType([
    StructField("entity_id", StringType(), False),
    StructField("entity_name", StringType(), False),
    StructField("entity_type", StringType(), False),
    StructField("region", StringType(), False),
    StructField("country", StringType(), False)
])

entities_data = [
    ("AXA-FR-001", "AXA France", "Insurance", "Europe", "France"),
    ("AXA-FR-002", "AXA France IARD", "Insurance", "Europe", "France"),
    ("AXA-DE-001", "AXA Germany", "Insurance", "Europe", "Germany"),
    ("AXA-UK-001", "AXA UK", "Insurance", "Europe", "United Kingdom"),
    ("AXA-BE-001", "AXA Belgium", "Insurance", "Europe", "Belgium"),
    ("AXA-CH-001", "AXA Switzerland", "Insurance", "Europe", "Switzerland"),
    ("AXA-IT-001", "AXA Italy", "Insurance", "Europe", "Italy"),
    ("AXA-ES-001", "AXA Spain", "Insurance", "Europe", "Spain"),
    ("AXA-US-001", "AXA XL US", "Reinsurance", "Americas", "United States"),
    ("AXA-US-002", "AXA Equitable", "Life Insurance", "Americas", "United States"),
    ("AXA-MX-001", "AXA Mexico", "Insurance", "Americas", "Mexico"),
    ("AXA-BR-001", "AXA Brazil", "Insurance", "Americas", "Brazil"),
    ("AXA-JP-001", "AXA Japan", "Insurance", "Asia Pacific", "Japan"),
    ("AXA-HK-001", "AXA Hong Kong", "Insurance", "Asia Pacific", "Hong Kong"),
    ("AXA-SG-001", "AXA Singapore", "Insurance", "Asia Pacific", "Singapore"),
    ("AXA-XL-001", "AXA XL Reinsurance", "Reinsurance", "Global", "Bermuda"),
    ("AXA-IM-001", "AXA Investment Managers", "Asset Management", "Global", "France"),
    ("AXA-GO-001", "AXA Global Operations", "Operations", "Global", "France"),
]

df_entities = spark.createDataFrame(entities_data, entities_schema)

# Save to ADLS as JSON
df_entities.write.mode("overwrite").json(ENTITIES_PATH)

print(f"Entities: {df_entities.count()} records")
print(f"Saved to: {ENTITIES_PATH}")
display(df_entities)

---

## 2. Generate Investment Placements - Initial Load

Main placement data for batch ingestion in Bronze Layer Lab.

In [ ]:
# Get entity IDs for random assignment
entity_ids = [row.entity_id for row in df_entities.collect()]

# Asset classes for placements
asset_classes = ["EQUITY", "BOND", "REAL_ESTATE", "ALTERNATIVE", "CASH", "DERIVATIVES"]
currencies = ["EUR", "USD", "GBP", "CHF"]

print(f"Entities: {len(entity_ids)}")
print(f"Asset Classes: {asset_classes}")

In [ ]:
# Generate initial placements data (300 records)

NUM_PLACEMENTS = 300

df_placements_base = spark.range(1, NUM_PLACEMENTS + 1).withColumnRenamed("id", "row_num")

df_placements = df_placements_base \
    .withColumn("placement_id", concat(lit("PLC-"), lpad(col("row_num").cast("string"), 6, "0"))) \
    .withColumn("axa_entity_id", element_at(array(*[lit(x) for x in entity_ids]), (floor(rand() * len(entity_ids)) + 1).cast("int"))) \
    .withColumn("asset_class", element_at(array(*[lit(x) for x in asset_classes]), (floor(rand() * len(asset_classes)) + 1).cast("int"))) \
    .withColumn("instrument_id", concat(lit("ISIN-"), lpad((col("row_num") % 100 + 1).cast("string"), 4, "0"))) \
    .withColumn("market_value", spark_round(rand() * 9000000 + 1000000, 2)) \
    .withColumn("book_value", spark_round(col("market_value") * (rand() * 0.3 + 0.85), 2)) \
    .withColumn("currency", element_at(array(*[lit(x) for x in currencies]), (floor(rand() * len(currencies)) + 1).cast("int"))) \
    .withColumn("valuation_date", date_add(lit("2024-01-01"), (rand() * 365).cast("int"))) \
    .select(
        "placement_id", "axa_entity_id", "asset_class", "instrument_id",
        "market_value", "book_value", "currency",
        col("valuation_date").cast("string")
    )

# Save initial batch to ADLS as JSON
df_placements.write.mode("overwrite").json(PLACEMENTS_PATH)

print(f"Placements (initial): {df_placements.count()} records")
print(f"Saved to: {PLACEMENTS_PATH}")
display(df_placements.limit(10))

---

## 3. Generate Incremental Placement Data

Used for Auto Loader demo in Bronze Layer Lab.

In [ ]:
# Generate incremental batch 1 (50 records)

NUM_INCREMENTAL_1 = 50

df_inc1_base = spark.range(301, 301 + NUM_INCREMENTAL_1).withColumnRenamed("id", "row_num")

df_placements_inc1 = df_inc1_base     .withColumn("placement_id", concat(lit("PLC-"), lpad(col("row_num").cast("string"), 6, "0")))     .withColumn("axa_entity_id", element_at(array(*[lit(x) for x in entity_ids]), (floor(rand() * len(entity_ids)) + 1).cast("int")))     .withColumn("asset_class", element_at(array(*[lit(x) for x in asset_classes]), (floor(rand() * len(asset_classes)) + 1).cast("int")))     .withColumn("instrument_id", concat(lit("ISIN-"), lpad((col("row_num") % 100 + 1).cast("string"), 4, "0")))     .withColumn("market_value", spark_round(rand() * 9000000 + 1000000, 2))     .withColumn("book_value", spark_round(col("market_value") * (rand() * 0.3 + 0.85), 2))     .withColumn("currency", element_at(array(*[lit(x) for x in currencies]), (floor(rand() * len(currencies)) + 1).cast("int")))     .withColumn("valuation_date", date_add(lit("2024-06-01"), (rand() * 90).cast("int")))     .select(
        "placement_id", "axa_entity_id", "asset_class", "instrument_id",
        "market_value", "book_value", "currency",
        col("valuation_date").cast("string")
    )

# Clear existing batch_1 data to avoid duplicates on re-run
try:
    dbutils.fs.rm(f"{PLACEMENTS_INCREMENTAL_PATH}/batch_1", recurse=True)
except:
    pass

# Save with fixed name (idempotent)
df_placements_inc1.coalesce(1).write.mode("overwrite").json(f"{PLACEMENTS_INCREMENTAL_PATH}/batch_1")

print(f"Incremental batch 1: {df_placements_inc1.count()} records")
print(f"Saved to: {PLACEMENTS_INCREMENTAL_PATH}/batch_1")

In [ ]:
# Generate incremental batch 2 (30 records)

NUM_INCREMENTAL_2 = 30

df_inc2_base = spark.range(351, 351 + NUM_INCREMENTAL_2).withColumnRenamed("id", "row_num")

df_placements_inc2 = df_inc2_base     .withColumn("placement_id", concat(lit("PLC-"), lpad(col("row_num").cast("string"), 6, "0")))     .withColumn("axa_entity_id", element_at(array(*[lit(x) for x in entity_ids]), (floor(rand() * len(entity_ids)) + 1).cast("int")))     .withColumn("asset_class", element_at(array(*[lit(x) for x in asset_classes]), (floor(rand() * len(asset_classes)) + 1).cast("int")))     .withColumn("instrument_id", concat(lit("ISIN-"), lpad((col("row_num") % 100 + 1).cast("string"), 4, "0")))     .withColumn("market_value", spark_round(rand() * 9000000 + 1000000, 2))     .withColumn("book_value", spark_round(col("market_value") * (rand() * 0.3 + 0.85), 2))     .withColumn("currency", element_at(array(*[lit(x) for x in currencies]), (floor(rand() * len(currencies)) + 1).cast("int")))     .withColumn("valuation_date", date_add(lit("2024-09-01"), (rand() * 60).cast("int")))     .select(
        "placement_id", "axa_entity_id", "asset_class", "instrument_id",
        "market_value", "book_value", "currency",
        col("valuation_date").cast("string")
    )

# Clear existing batch_2 data to avoid duplicates on re-run
try:
    dbutils.fs.rm(f"{PLACEMENTS_INCREMENTAL_PATH}/batch_2", recurse=True)
except:
    pass

# Save with fixed name (idempotent)
df_placements_inc2.coalesce(1).write.mode("overwrite").json(f"{PLACEMENTS_INCREMENTAL_PATH}/batch_2")

print(f"Incremental batch 2: {df_placements_inc2.count()} records")
print(f"Saved to: {PLACEMENTS_INCREMENTAL_PATH}/batch_2")

---

## 4. Verification Summary

In [ ]:
# Verification summary

print("="*70)
print("DATA GENERATION SUMMARY")
print("="*70)

datasets = [
    ("AXA Entities (Reference)", ENTITIES_PATH, "JSON"),
    ("Placements Initial (Bronze - Batch)", PLACEMENTS_PATH, "JSON"),
    ("Placements Incremental (Bronze - Auto Loader)", PLACEMENTS_INCREMENTAL_PATH, "JSON"),
]

for name, path, fmt in datasets:
    try:
        files = dbutils.fs.ls(path)
        file_count = len([f for f in files if not f.name.startswith("_")])
        print(f"\n{name}")
        print(f"  Path: {path}")
        print(f"  Format: {fmt}")
        print(f"  Items: {file_count}")
    except Exception as e:
        print(f"\n{name}: ERROR - {e}")

print("\n" + "="*70)
print("DATA GENERATION COMPLETE")
print("="*70)

---

## Data Paths Reference

Use these paths in Day 2 lab notebooks:

| Dataset | Path Pattern | Lab |
|---------|-------------|-----|
| Entities | `{BASE_PATH}/raw/entities` | Silver Layer - Enrichment |
| Placements Initial | `{BASE_PATH}/raw/placements` | Bronze Layer - Batch |
| Placements Incremental | `{BASE_PATH}/raw/placements_incremental` | Bronze Layer - Auto Loader |

---

## Next Steps

Proceed to the Day 2 lab notebooks:

1. **`jour2/1_E2E_Bronze_Layer.ipynb`** - Raw data ingestion with Auto Loader
2. **`jour2/2_E2E_Silver_Layer.ipynb`** - Data cleaning and enrichment
3. **`jour2/3_E2E_Gold_Layer.ipynb`** - Aggregations and optimization